# BirdCLEF+ 2025 — SED inference (Member 2, submission notebook)

Runs under the competition limits: **CPU only, internet off, 90 minutes**. Loads `model.pth`,
`cfg.json` and `species.json` from the training notebook and writes `submission.csv`.

**How predictions are made:**
1. Make one spectrogram for the whole 60 s recording.
2. Cut it into 20 s chunks, `STRIDE_SEC` apart, and scale each chunk by its own loudest point, as in training.
3. The model scores every moment of each chunk.
4. For each 5 s window, take the highest score per species. Where chunks overlap, average their answers.

The hidden test set only appears when Kaggle scores a submission. In a normal run
`test_soundscapes/` is empty, so we do a dry run on 20 unlabelled soundscapes and estimate the full time.

Files are located by a search that skips the audio folders, and every stage is timed.

In [ ]:
STRIDE_SEC = 5   # seconds between chunks: 20 = no overlap, 5 = heavy overlap

In [ ]:
import time
t_start = time.time()
import os, json
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import timm
print(f"Imports: {time.time() - t_start:.1f}s")


INPUT = Path("/kaggle/input")
SKIP = {"train_audio", "train_soundscapes", "test_soundscapes"}

def find_dir(marker, max_depth=4):
    """Find the folder holding `marker`, without walking through the 38,000 audio files."""
    for folder, subdirs, files in os.walk(INPUT):
        depth = len(Path(folder).relative_to(INPUT).parts)
        subdirs[:] = [d for d in subdirs if d not in SKIP] if depth < max_depth else []
        if marker in files:
            return Path(folder)
    raise FileNotFoundError(f"{marker} not found under {INPUT}")

t0 = time.time()
DATA, MODEL_DIR = find_dir("sample_submission.csv"), find_dir("model.pth")
print(f"File search: {time.time() - t0:.1f}s | data: {DATA} | model: {MODEL_DIR}")

CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"

torch.set_num_threads(os.cpu_count())
SR, HOP = CFG["sr"], CFG["hop_length"]
CHUNK_SEC, WINDOW_SEC = CFG["chunk_sec"], CFG["window_sec"]
FRAMES = CFG["frames_per_chunk"]
WINDOWS_PER_CHUNK = CHUNK_SEC // WINDOW_SEC
STORE_DB, TOP_DB = 120.0, 80.0
assert STRIDE_SEC % WINDOW_SEC == 0, "chunks must start on a 5 s window boundary"
print(f"Experiment {CFG['experiment']} | chunk {CHUNK_SEC}s, stride {STRIDE_SEC}s | CPUs: {os.cpu_count()}")

In [ ]:
test_files = sorted((DATA / "test_soundscapes").glob("*.ogg"))
DRY_RUN = len(test_files) == 0
if DRY_RUN:
    test_files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:20]
print(f"{'DRY RUN on' if DRY_RUN else 'Predicting'} {len(test_files)} soundscapes")

## Spectrograms
The settings come from the training notebook's `cfg.json`, so the model sees what it saw in training.
Each recording is done once, however many chunks are later cut from it.

In [ ]:
def recording_mel(path):
    y, _ = sf.read(path, dtype="float32", always_2d=True)
    y = y.mean(axis=1)
    samples_per_window = SR * WINDOW_SEC
    n_windows = max(1, int(np.ceil(len(y) / samples_per_window)))
    y = np.pad(y, (0, n_windows * samples_per_window - len(y)))
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=CFG["n_fft"], hop_length=HOP,
                                         n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    return librosa.power_to_db(mel, ref=np.max, top_db=STORE_DB).astype(np.float32), n_windows

t0 = time.time()
with Pool(os.cpu_count()) as pool:
    recordings = pool.map(recording_mel, test_files)
mel_sec = time.time() - t0
print(f"Spectrograms: {mel_sec:.0f}s for {len(test_files)} recordings")

## The model
Exactly the same model definition as the training notebook.

In [ ]:
class SEDModel(nn.Module):
    """EfficientNet that keeps the time axis, so it can score every moment of the chunk."""

    def __init__(self, cfg, n_classes, pretrained):
        super().__init__()
        self.backbone = timm.create_model(cfg["model"], pretrained=pretrained, in_chans=1,
                                          num_classes=0, global_pool="")
        channels, hidden, drop = self.backbone.num_features, cfg["head_dim"], cfg["dropout"]
        self.gem_p = cfg["gem_p"]
        self.frame_net = nn.Sequential(nn.Dropout(drop), nn.Conv1d(channels, hidden, 1),
                                       nn.ReLU(inplace=True), nn.Dropout(drop))
        self.species_scores = nn.Conv1d(hidden, n_classes, 1)
        self.attention = nn.Conv1d(hidden, n_classes, 1)
        nn.init.constant_(self.species_scores.bias, -4.0)  # start with "probably not calling"

    def pool_frequency(self, features):
        """GeM pooling: (batch, channels, freq, time) -> (batch, channels, time)."""
        positive = features.float().clamp(min=1e-4)
        return positive.pow(self.gem_p).mean(dim=2).pow(1.0 / self.gem_p)

    def forward(self, mel):
        features = self.backbone.forward_features(mel)
        hidden = self.frame_net(self.pool_frequency(features))
        frame_logits = self.species_scores(hidden)                 # (batch, species, time)
        weights = torch.softmax(self.attention(hidden), dim=-1)    # which moments matter
        clip_logits = (weights * frame_logits).sum(dim=-1)         # attention pooling
        peak_logits = frame_logits.max(dim=-1).values              # loudest moment per species
        return frame_logits, clip_logits, peak_logits

In [ ]:
model = SEDModel(CFG, len(species), pretrained=False)
model.load_state_dict(torch.load(MODEL_DIR / "model.pth", map_location="cpu"))
model.eval()

## Predict
Each time step is assigned to the 5 s window its centre falls in. Each chunk gives every window it
covers one score per species, which is the highest score among that window's time steps.

In [ ]:
def chunk_starts(n_windows):
    """Start times in seconds. Always include a final chunk that reaches the end of the recording."""
    last = max(0, n_windows - WINDOWS_PER_CHUNK) * WINDOW_SEC
    starts = list(range(0, last + 1, STRIDE_SEC))
    return starts if starts[-1] == last else starts + [last]

def to_input(db):
    """0 dB = this chunk's loudest point, keep 80 dB below it, scale to [0, 1]."""
    return ((np.maximum(db - db.max(), -TOP_DB) + TOP_DB) / TOP_DB).astype(np.float32)

def cut_chunk(db, start_sec):
    first = int(round(start_sec * SR / HOP))
    chunk = db[:, first : first + FRAMES]
    if chunk.shape[1] < FRAMES:
        chunk = np.pad(chunk, ((0, 0), (0, FRAMES - chunk.shape[1])), constant_values=-STORE_DB)
    return to_input(chunk)

@torch.inference_mode()
def predict_chunks(chunks):
    frame_logits = model(torch.from_numpy(np.stack(chunks))[:, None])[0]
    return torch.sigmoid(frame_logits).numpy()              # (chunks, species, time)

t0 = time.time(); row_ids, rows, n_chunks = [], [], 0
for path, (db, n_windows) in zip(test_files, recordings):
    starts = chunk_starts(n_windows)
    total, count = np.zeros((n_windows, len(species))), np.zeros(n_windows)
    for i in range(0, len(starts), 16):
        batch = starts[i : i + 16]
        probs = predict_chunks([cut_chunk(db, s) for s in batch])
        T = probs.shape[-1]
        centre_sec = (np.arange(T) + 0.5) * CHUNK_SEC / T
        window_of_step = np.minimum((centre_sec // WINDOW_SEC).astype(int), WINDOWS_PER_CHUNK - 1)
        for s, p in zip(batch, probs):
            first_window = s // WINDOW_SEC
            for w in range(WINDOWS_PER_CHUNK):
                if first_window + w < n_windows:
                    total[first_window + w] += p[:, window_of_step == w].max(axis=1)
                    count[first_window + w] += 1
        n_chunks += len(batch)
    assert (count > 0).all(), f"{path.name}: a window received no prediction"
    rows.append(total / count[:, None])
    row_ids += [f"{path.stem}_{(w + 1) * WINDOW_SEC}" for w in range(n_windows)]
predict_sec = time.time() - t0
print(f"Model: {predict_sec:.0f}s for {n_chunks} chunks | {T} time steps per chunk")

In [ ]:
sub = pd.DataFrame(np.concatenate(rows), columns=species)
sub.insert(0, "row_id", row_ids)
assert not sub.isna().any().any(), "submission contains missing values"
assert list(sub.columns) == list(sample_sub.columns), "columns differ from sample_submission"
assert sub["row_id"].is_unique, "duplicate row ids"
sub.to_csv("submission.csv", index=False)

n = len(test_files); total_sec = time.time() - t_start
estimate = (mel_sec + predict_sec) / n * 700 / 60
print(f"submission.csv: {sub.shape} | first rows: {list(sub['row_id'][:3])} | last row: {sub['row_id'].iloc[-1]}")
print(f"Whole notebook: {total_sec / 60:.1f} min for {n} recordings")
print(f"Estimate for about 700 test recordings: {estimate:.0f} min on this machine (limit 90)")
sub.iloc[:5, :6]